# Export your trained VGG16 Cats vs. Dogs model

This short notebook reproduces the **VGG16 transfer-learning experiment** from `transfer-learning_true.ipynb`, evaluates the newly trained model, and exports the real trained weights to `cats_dogs_vgg16.keras`.

Use **Runtime → Change runtime type → T4 GPU** in Colab. Run **all cells in order**, not just the final export cell. The 97.56% test accuracy previously recorded in the original notebook is not guaranteed after retraining.


In [ ]:
!pip -q install tensorflow-datasets scikit-learn

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_datasets as tfds
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))

## Step 1 — Load the same 3,000-image subset and create the same split

In [ ]:
raw_ds, ds_info = tfds.load("cats_vs_dogs", split="train", with_info=True, as_supervised=True)
print("Dataset classes:", ds_info.features["label"].names)  # ['cat', 'dog']

N_PER_CLASS = 1500
raw_balanced_ds = raw_ds.filter(lambda image, label: label == 0).take(N_PER_CLASS).concatenate(
    raw_ds.filter(lambda image, label: label == 1).take(N_PER_CLASS)
)
TOTAL_IMAGES = N_PER_CLASS * 2
N_TRAIN = 2100
N_VAL = 450
N_TEST = 450
shuffled_ds = raw_balanced_ds.shuffle(
    buffer_size=TOTAL_IMAGES, seed=SEED, reshuffle_each_iteration=False
)
train_raw_ds = shuffled_ds.take(N_TRAIN)
val_raw_ds = shuffled_ds.skip(N_TRAIN).take(N_VAL)
test_raw_ds = shuffled_ds.skip(N_TRAIN + N_VAL).take(N_TEST)
print("train / validation / test:", N_TRAIN, N_VAL, N_TEST)

## Step 2 — Match the notebook preprocessing exactly

Resize with TensorFlow then call `keras.applications.vgg16.preprocess_input`. **Do not divide pixels by 255.**

In [ ]:
IMG_SIZE_VGG = (224, 224)
BATCH_SIZE = 32

def vgg_preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE_VGG)
    image = keras.applications.vgg16.preprocess_input(image)
    return image, label

vgg_train_ds = (
    train_raw_ds.map(vgg_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .ignore_errors().shuffle(N_TRAIN, seed=SEED)
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
)
vgg_val_ds = (
    val_raw_ds.map(vgg_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .ignore_errors().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
)
vgg_test_ds = (
    test_raw_ds.map(vgg_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .ignore_errors().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
)

## Step 3 — Build and train the original VGG16 transfer-learning model

Frozen pretrained backbone; fresh small classifier head; 8 epochs.

In [ ]:
vgg_tl_base = keras.applications.VGG16(
    include_top=False, weights="imagenet", input_shape=(224, 224, 3)
)
vgg_tl_base.trainable = False
vgg_tl_model = keras.Sequential([
    vgg_tl_base,
    layers.Flatten(),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
], name="vgg_transfer")
vgg_tl_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
vgg_tl_model.summary()

In [ ]:
history = vgg_tl_model.fit(
    vgg_train_ds, validation_data=vgg_val_ds, epochs=8, verbose=2
)

## Step 4 — Evaluate on the held-out test set

In [ ]:
test_loss, test_acc = vgg_tl_model.evaluate(vgg_test_ds, verbose=0)
y_test = np.concatenate([y for _, y in tfds.as_numpy(vgg_test_ds)])
y_prob = vgg_tl_model.predict(vgg_test_ds, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)
print(f"Test loss: {test_loss:.4f}, test accuracy: {test_acc:.2%}")
print(classification_report(y_test, y_pred, target_names=["cat", "dog"], digits=3))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

## Step 5 — Export trained weights and verify model loading

**Do not rebuild/reinitialize the model between training and this cell.**

In [ ]:
from pathlib import Path

model_file = Path("cats_dogs_vgg16.keras")
vgg_tl_model.save(str(model_file))
reloaded_model = keras.models.load_model(str(model_file), compile=False)

sample_x, _ = next(iter(vgg_test_ds))
pred_original = vgg_tl_model(sample_x, training=False).numpy()
pred_reloaded = reloaded_model(sample_x, training=False).numpy()
np.testing.assert_allclose(pred_original, pred_reloaded, rtol=1e-5, atol=1e-6)
print(f"✅ Saved and verified {model_file} ({model_file.stat().st_size / 1024**2:.1f} MiB)")

## Step 6 — Download and add to the Streamlit app

Download the file and place it at `cats-dogs-live-demo/models/cats_dogs_vgg16.keras`. Then run `streamlit run app.py`.

In [ ]:
try:
    from google.colab import files
    files.download("cats_dogs_vgg16.keras")
except ImportError:
    print("Not in Colab. Download the file from the current working directory.")